# CS604 — Qian’s subset BitFit ablation at 336

Run only the LR pilot and five-arm scope screen on the same 1,000 classes, three epochs each. The matching all-bias pilot run is reused. Fit-only prototypes, fixed 40/5 split, seed 42, and a 10,000-way linear head. Use an A100 GPU. Results and completion markers persist on Drive.

After selecting a scope and LR, return to qq_branch_v2 for a separate fresh all-data run. Publish this branch before cloning in Colab.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "qq_bitfit_ablation"
TEAM = "08"
CONFIG = "configs/bitfit_ablation_336.yaml"
# Reuse this name across sessions. Change it when changing the recipe.
EXPERIMENT_NAME = "qq_bitfit_ablation_336_subset_seed42_v2"
EXPERIMENT_DIR = f"{DRIVE_OUT}/{EXPERIMENT_NAME}"
SPLIT_DIR = f"/content/{EXPERIMENT_NAME}_splits"
RUN_PILOT = True
RUN_SCREEN = True
BIAS_LR = None  # None uses the persisted pilot choice; explicit override starts a new recipe.
ARM_FILTER = None  # None runs all 5; e.g. ["head_only", "all_biases"] for one session.


## 1. Clone or pull the repo

In [ ]:
import base64
import os
import subprocess
from getpass import getpass

# Pass credentials through a temporary child-process environment, not the URL.
token = getpass("GitHub PAT (hidden; repo scope). Leave empty for a public repo: ")
git_env = os.environ.copy()
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    del auth
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], env=git_env, check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], env=git_env, check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], env=git_env, check=True)
finally:
    del token, git_env
os.chdir(REPO_DIR)
actual = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
assert actual == BRANCH, f"Wrong branch: {actual}"
print("Branch:", actual)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
subprocess.run(["python", "scripts/download_data.py", "--dest", DATA_DIR], check=True)


## 4b. Prepare the fixed split

Both stages use the same 1,000 classes: 40,000 fitting and 5,000 holdout images. The head stays 10,000-way. Prototypes use only fitting images; absent classes have zero rows. Full split manifests are saved for provenance, but no full-scale training runs here.


In [ ]:
import json
import subprocess
from pathlib import Path

import pandas as pd
import torch
from src.utils import load_config, apply_overrides
from src.peft.bitfit_ablation import prepare, arm_grid, run_stage

assert torch.cuda.is_available(), "Select a GPU runtime before running the experiment."
BASE = load_config(CONFIG)
apply_overrides(BASE, data_root=DATA_DIR, output_dir=EXPERIMENT_DIR)
assert BASE.img_size == 336 and BASE.head_init == "prototype" and BASE.head_type == "linear"
assert BASE.max_train_minutes is None and BASE.epochs == 3
EXPERIMENT = Path(EXPERIMENT_DIR)
ROOTS = prepare(BASE, EXPERIMENT, split_dir=SPLIT_DIR)
print("GPU:", torch.cuda.get_device_name(0))
print("Experiment:", EXPERIMENT)
print("Split identity:", json.loads((EXPERIMENT / "experiment.json").read_text()))
display(pd.DataFrame(list(BASE.to_dict().items()), columns=["parameter", "value"]))
from src.model import build_model
from src.utils import Config
rows = []
for arm in arm_grid(BASE):
    check = Config(BASE.to_dict())
    check.update(arm)
    check.pretrained = False
    model = build_model(check)
    head_ids = {id(p) for p in model.get_classifier().parameters()}
    backbone = sum(p.numel() for p in model.parameters() if id(p) not in head_ids)
    count = sum(p.numel() for p in model.parameters() if id(p) not in head_ids and p.requires_grad)
    rows.append({**arm, "bias_parameters": count, "backbone_trainable_pct": 100 * count / backbone})
    del model
display(pd.DataFrame(rows))
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
(EXPERIMENT / "launcher_commit.txt").write_text(commit + "\n")


## 5. Learning-rate pilot

Three 3-epoch runs with all backbone biases: LR 1e-4, 3e-4, 1e-3. Head LR stays 1e-3. Choose final holdout accuracy, preferring the lower stable LR within 0.3 points. All arms use AdamW, zero weight decay, BF16, 5% warmup and cosine decay.


In [ ]:
if RUN_PILOT:
    PILOT_RESULTS = run_stage(BASE, EXPERIMENT, ROOTS, "pilot")
    display(pd.DataFrame(PILOT_RESULTS))
if (EXPERIMENT / "pilot_selection.json").exists():
    print("Pilot selection:", json.loads((EXPERIMENT / "pilot_selection.json").read_text()))


## 6. Scope screen

Compare all five scopes on the same 1,000 classes for three epochs. The matching all-bias pilot run is reused after recipe validation, leaving four new runs. Rank by final holdout accuracy. ARM_FILTER supports selected arms across sessions.


In [ ]:
SCREEN_RESULTS = []
if RUN_SCREEN:
    SCREEN_RESULTS = run_stage(BASE, EXPERIMENT, ROOTS, "screen", arms=ARM_FILTER, bias_lr=BIAS_LR)
summary = EXPERIMENT / "screen_summary.json"
SCREEN_RESULTS = json.loads(summary.read_text()) if summary.exists() else []
if SCREEN_RESULTS:
    SCREEN_TABLE = pd.DataFrame(SCREEN_RESULTS).sort_values("final_val_acc", ascending=False)
    display(SCREEN_TABLE)
    if len(SCREEN_TABLE) == 5:
        best = SCREEN_TABLE.iloc[0]
        print("Best screen arm:", best["arm"], "bias LR:", best["lr"])
    else:
        print("Partial screen: finish all five arms before selection.")


## 8. Loss curves

Plot the completed screen arms. Validation curves use only the internal holdout. Compare initial/best/final accuracy and loss; five epochs do not establish convergence.


In [ ]:
from IPython.display import display, Image

for result in SCREEN_RESULTS:
    run_dir = Path(result["run_dir"])
    figures = run_dir / "figures"
    subprocess.run(["python", "scripts/plot_curves.py", "--run_dir", str(run_dir),
                    "--out_dir", str(figures)], check=True)
    print(result["arm"])
    for plot in sorted(figures.glob("*.png")):
        display(Image(filename=str(plot)))


## Next step

Review the completed scope screen and select a bias scope and LR. Return to `qq_branch_v2`, carry over the selected scope support, and prepare a fresh all-data 144-minute recipe. Do not reuse an ablation checkpoint or subset prototypes for that run.
